# **Memory**

# Install Libraries

In [ ]:
!pip install -q crewai crewai_tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [ ]:
!pip install -q sentence-transformers

In [ ]:
!pip install -q -U --force-reinstall pillow

# Version Check

After successful installation....

In [ ]:
import crewai
import crewai_tools
print(crewai.__version__)
print(crewai_tools.__version__)

1.15.14
1.15.14


# Memory

Leveraging the unified memory system in CrewAI to enhance agent capabilities.

​
## Overview

CrewAI provides a unified memory system — a single Memory class that replaces separate short-term, long-term, entity, and external memory types with one intelligent API. Memory uses an LLM to analyze content when saving (inferring scope, categories, and importance) and supports adaptive-depth recall with composite scoring that blends semantic similarity, recency, and importance.

You can use memory four ways: **standalone** (scripts, notebooks), with **Crews**, with **Agents**, or inside **Flows**.

In [ ]:
from crewai import Memory

memory = Memory()

# Store -- the LLM infers scope, categories, and importance
memory.remember("We decided to use PostgreSQL for the user database.")

# Retrieve -- results ranked by composite score (semantic + recency + importance)
matches = memory.recall("What database did we choose?")
for m in matches:
    print(f"[{m.score:.2f}] {m.record.content}")

# Tune scoring for a fast-moving project
memory = Memory(recency_weight=0.5, recency_half_life_days=7)

# Explore the self-organized scope tree
print(memory.tree())
print(memory.info("/"))

In [ ]:
# Forget
memory.forget(scope="/project/old")

# Explore the self-organized scope tree
print(memory.tree())

print(memory.info("/"))

# Standalone

Use memory in scripts, notebooks, CLI tools, or as a standalone knowledge base — no agents or crews required.

In [ ]:
from crewai import Memory

memory = Memory()

# Build up knowledge
memory.remember("The API rate limit is 1000 requests per minute.")
memory.remember("Our staging environment uses port 8080.")
memory.remember("The team agreed to use feature flags for all new releases.")

# Later, recall what you need
matches = memory.recall("What are our API limits?", limit=5)
for m in matches:
    print(f"[{m.score:.2f}] {m.record.content}")

# Extract atomic facts from a longer text
raw = """Meeting notes: We decided to migrate from MySQL to PostgreSQL
next quarter. The budget is $50k. Sarah will lead the migration."""

facts = memory.extract_memories(raw)
# ["Migration from MySQL to PostgreSQL planned for next quarter",
#  "Database migration budget is $50k",
#  "Sarah will lead the database migration"]

for fact in facts:
    memory.remember(fact)

# Set API Keys for LLMs & Tools

In [ ]:
from google.colab import userdata
import os
# os.environ["SERPER_API_KEY"] = userdata.get('SERPER_API_KEY')
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY_NEW')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
# os.environ["HUGGINGFACE_API_KEY"] = userdata.get('HF_TOKEN_NEW')

os.environ["CREWAI_STORAGE_DIR"] = "./"

# Configure an LLM

### GROQ Llama-3.3-70b-versatile for the Agents
and
### GROQ OpenAI GPT-OSS-120b for Memory


In [ ]:
from crewai import LLM

# # GROQ - LLaMA-3.3-70B - UNABLE to handle structured output, not suitable
# agent_llm = LLM(
#     model="llama-3.3-70b-versatile",
#     base_url="https://api.groq.com/openai/v1",
#     api_key=os.environ["GROQ_API_KEY"],
#     temperature=0.7
# )


# # GROQ- QWEN-3.6-27B - needs to handle structured output from search_memory tool
# agent_llm = LLM(
#     model="qwen/qwen3.6-27b",
#     base_url="https://api.groq.com/openai/v1",
#     api_key=os.environ["GROQ_API_KEY"],
#     temperature=0.7
# )


# Gemini - needs to handle structured output from search_memory tool
agent_llm = LLM(
    model="gemini/gemini-3.1-flash-lite",
    # base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.7
)



# Memory LLM should be capable enough to handle structured output

In [ ]:
# # GROQ - GPT-OSS/QWEN
# memory_llm = LLM(
#     # model="openai/gpt-oss-120b",
#     base_url="https://api.groq.com/openai/v1",
#     api_key=os.environ["GROQ_API_KEY"],
#     temperature=0
# )

# Gemini
memory_llm = LLM(
    model="gemini/gemini-3.1-flash-lite",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0.7
)


# Configure an Embedding Model (HuggingFace)

In [ ]:
# from sentence_transformers import SentenceTransformer

# embedding_model = SentenceTransformer(
#     "sentence-transformers/all-MiniLM-L6-v2"
# )

# def my_embedder(texts: list[str]) -> list[list[float]]:
#     return embedding_model.encode(
#         texts,
#         normalize_embeddings=True
#     ).tolist()

In [ ]:
# from crewai import Memory
# memory = Memory(
#     llm=memory_llm,
#     # embedder=embed
#     embedder=my_embedder
# )

# Agent-level Memory

In [ ]:
import os
from crewai import Agent, Task, Crew, Process, Memory, LLM

# Create Memory

In [ ]:
memory = Memory(
    llm=memory_llm,
    embedder={"provider": "sentence-transformer", "config": {"model_name": "all-MiniLM-L6-v2"}},
)

# Initialize Memory with some facts

In [ ]:
memory.remember(
    content="""The software project is called SmartCampus.
    SmartCampus is implemented using Python.
    SmartCampus uses PostgreSQL as its database.
    The SmartCampus project has four team members.
    The SmartCampus project deadline is 30 September 2026.""",

    scope="/project/smartcampus"
)


╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 2019.06ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

MemoryRecord(id='0c03bb76-96c3-4537-b7e9-a204ad6bc3ba', content='The software project is called SmartCampus. \n    SmartCampus is implemented using Python. \n    SmartCampus uses PostgreSQL as its database. \n    The SmartCampus project has four team members.\n    The SmartCampus project deadline is 30 September 2026.', scope='/project/smartcampus', categories=[], metadata={'entities': [], 'dates': [], 'topics': []}, importance=0.5, created_at=datetime.datetime(2026, 8, 11, 5, 38, 38, 794530), last_accessed=datetime.datetime(2026, 8, 11, 5, 38, 38, 794546), source=None, private=False)

# Create Scoped Memory for the Project Advisor Agent

In [ ]:
advisor_memory = memory.scope("/project/smartcampus")

In [ ]:
# # Forget
# memory.forget(scope="/project/old")

# Explore the self-organized scope tree
print(advisor_memory.tree())

print(advisor_memory.info())
# print(advisor_memory.info("/")) # root of the tree

/project/smartcampus (4 records)
path='/project/smartcampus' record_count=4 categories=['Deadlines', 'Project Management', 'Software Development', 'Team Composition'] oldest_record=datetime.datetime(2026, 8, 11, 5, 4, 19, 561067) newest_record=datetime.datetime(2026, 8, 11, 5, 38, 38, 794530) child_scopes=[]


In [ ]:
matches = memory.recall("What is the expected project completion time?",limit=1)
# matches
m = matches[0]
print(f"[{m.score:.2f}] {m.record.content}")


[0.72] The expected completion date for the SmartCampus project is September 30, 2026.


In [ ]:
matches = memory.recall("What is the project team size?")
m = matches[0]
print(f"[{m.score:.2f}] {m.record.content}")


[0.67] The SmartCampus project team consists of four members.


In [ ]:

# There is no budget mentioned in the memory!

matches = memory.recall("What is the total budget for the project?")
m = matches[0]
print(f"[{m.score:.2f}] {m.record.content}")


[0.67] The expected completion date for the SmartCampus project is September 30, 2026.


# Create Agents

In [ ]:

advisor = Agent(
    role="Project Advisor",
    goal="Answer project-related questions using your available memory. {question}",
    backstory=(
        "You are an experienced software project advisor. "
        "Use your memory when answering questions."
    ),
    llm=agent_llm,
    memory=advisor_memory,
    verbose=True
)


# analyst = Agent(
#     role="Technical Analyst",
#     goal="Analyze software engineering issues.",
#     backstory="You are an experienced software engineering analyst.",
#     llm=agent_llm,
#     verbose=True
# )


# writer = Agent(
#     role="Technical Writer",
#     goal="Write concise technical summaries.",
#     backstory="You are an experienced technical writer.",
#     llm=agent_llm,
#     verbose=True
# )


## Memory of the Advisor Agent

In [ ]:
advisor.memory.info()

ScopeInfo(path='/project/smartcampus', record_count=4, categories=['Deadlines', 'Project Management', 'Software Development', 'Team Composition'], oldest_record=datetime.datetime(2026, 8, 11, 5, 4, 19, 561067), newest_record=datetime.datetime(2026, 8, 11, 5, 38, 38, 794530), child_scopes=[])

# Create Tasks

In [ ]:
# Task for the Advisor Agent
advisor_task = Task(
    description="""
    Answer the user question using ONLY information
    available in your memory.
    1. What is the name of the project?
    2. Which programming language is used?
    3. Which database is used?
    4. How many team members are there?
    5. What is the project deadline?

    Do not invent any information.
    """,
    expected_output="""
    A numbered list containing the five questions and
    their corresponding answers.
    """,
    agent=advisor
)


# analyst_task = Task(
#     description="""
#     Explain briefly why persistent memory can be useful
#     for a software project advisor working across multiple
#     interactions.
#     """,
#     expected_output="A short explanation.",
#     agent=analyst
# )


# writer_task = Task(
#     description="""
#     Write a short explanation of how agent-level memory
#     differs from ordinary task context.
#     """,
#     expected_output="A short explanation.",
#     agent=writer
# )


# Create Crew

In [ ]:

crew = Crew(
    agents=[
        advisor,
        # analyst,
        # writer
    ],
    tasks=[
        advisor_task,
        # analyst_task,
        # writer_task
    ],
    # process=Process.sequential,
    # memory=memory,
    verbose=True
)


# Run the Crew

In [ ]:
result = await crew.kickoff_async()

print("\n========== FINAL RESULT ==========\n")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1dad5c39-4aa3-41ef-8e1c-80fe045d37e2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Answer the user question using ONLY information                                                            │
│      available in your memory.                                                                                  │
│      1. What is the name of the project?                                                                        │
│      2. Which programming language is used?                                                                     │
│      3. Which database is used?                                                                                 │
│      4. How many team members are there?                                                                        │
│      5. What is the project deadline?                                                                           │
│                                                                                                                 │
│      Do not invent any information.                                                                             │
│                                                                                                                 │
│  ID: 5cb38457-f2b8-4163-b2e5-b906d9fd82b8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieval ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Started                                                                                       │
│  Status: Retrieving...                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 🧠 Memory Retrieved ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Retrieval Completed                                                                                     │
│  Time: 945.76ms                                                                                                 │
│  Content:                                                                                                       │
│  Relevant memories:                                                                                             │
│  - (score=0.71) The project is named SmartCampus.                                                               │
│    categories: Project Management                                                                               │
│    entities: ['SmartCampus']                                                                                    │
│    dates: []                                                                                                    │
│    topics: ['Project Naming']                                                                                   │
│  - (score=0.67) The expected completion date for the SmartCampus project is September 30, 2026.                 │
│    categories: Project Management, Deadlines                                                                    │
│    entities: ['SmartCampus']                                                                                    │
│    dates: ['2026-09-30']                                                                                        │
│    topics: ['Project completion', 'Timeline']                                                                   │
│  - (score=0.64) The SmartCampus project team consists of four members.                                          │
│    categories: Project Management, Team Composition                                                             │
│    entities: ['SmartCampus']                                                                                    │
│    dates: []                                                                                                    │
│    topics: ['Team Structure', 'Project Management']                                                             │
│  - (score=0.62) The SmartCampus project uses PostgreSQL as its database.                                        │
│    categories: Software Development                                                                             │
│    entities: ['SmartCampus', 'PostgreSQL']                                                                      │
│    dates: []                                                                                                    │
│    topics: ['Database Infrastructure', 'Technology Stack']                                                      │
│  - (score=0.62) The software project is called SmartCampus.                                                     │
│      SmartCampus is implemented using Python.                                                                   │
│      SmartCampus uses PostgreSQL as its database.                                                               │
│      The SmartCampus project has four team members.                                                             │
│      The SmartCampus project deadline is 30 September 2026.                                                     │
│    entities: []                                                                                                 │
│    dates: []                                                                                                    │
│    topics: []                                                                                                   │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Project Advisor                                                                                         │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Answer the user question using ONLY information                                                            │
│      available in your memory.                                                                                  │
│      1. What is the name of the project?                                                                        │
│      2. Which programming language is used?                                                                     │
│      3. Which database is used?                                                                                 │
│      4. How many team members are there?                                                                        │
│      5. What is the project deadline?                                                                           │
│                                                                                                                 │
│      Do not invent any information.                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── 🧠 Memory Save ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Started                                                                                            │
│  Status: Saving...                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── ✅ Memory Saved ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Memory Save Completed                                                                                          │
│  Source: Unified Memory                                                                                         │
│  Time: 2195.30ms                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1dad5c39-4aa3-41ef-8e1c-80fe045d37e2                                                                       │
│  Final Output: 1. What is the name of the project? SmartCampus                                                  │
│  2. Which programming language is used? Python                                                                  │
│  3. Which database is used? PostgreSQL                                                                          │
│  4. How many team members are there? Four                                                                       │
│  5. What is the project deadline? September 30, 2026                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


========== FINAL RESULT ==========

1. What is the name of the project? SmartCampus
2. Which programming language is used? Python
3. Which database is used? PostgreSQL
4. How many team members are there? Four
5. What is the project deadline? September 30, 2026




╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────


# Crew-Level Memory

Here the same Groq LLM is used by all three agents, but the memory is attached to the Crew rather than to an individual agent.

Pass **memory=True** for default settings, or pass a configured Memory instance for custom behavior.




In [ ]:
import os

from crewai import Agent, Task, Crew, Process, Memory, LLM


# ============================================================
# Create shared Memory
# ============================================================

customer_memory = Memory(
    llm=memory_llm,
    embedder=embed
)


# ============================================================
# Initialize Memory with existing facts
# ============================================================

customer_memory.remember(
    content = """
    "Acme Corporation is an enterprise customer.
    Acme Corporation has 50 licensed users.",
    Acme Corporation prefers email communication.",
    Enterprise customers have an API rate limit of 1000 requests per minute."
    """,
    scope="/product"
)


# ============================================================
# Create Agents
# ============================================================

support_agent = Agent(
    role="Customer Support Agent",
    goal="Understand customer requirements and provide accurate answers.",
    backstory="You are an experienced customer support specialist.",
    llm=agent_llm,
    verbose=True
)


technical_agent = Agent(
    role="Technical Support Agent",
    goal="Answer technical questions using available information.",
    backstory="You are an experienced technical support engineer.",
    llm=agent_llm,
    verbose=True
)


manager_agent = Agent(
    role="Support Manager",
    goal="Review customer information and prepare a final response.",
    backstory="You manage enterprise customer support operations.",
    llm=agent_llm,
    verbose=True
)


# ============================================================
# Create Tasks
# ============================================================

support_task = Task(
    description="""
    Using the crew's shared memory, answer:

    1. What type of customer is Acme Corporation?
    2. How many licensed users does Acme have?
    3. What communication method does Acme prefer?

    Do not invent information.
    """,
    expected_output="""
    Three questions with their corresponding answers.
    """,
    agent=support_agent
)


technical_task = Task(
    description="""
    Using the crew's shared memory, answer:

    What is the API rate limit for an enterprise customer?

    Do not use external information.
    """,
    expected_output="The API rate limit with a brief explanation.",
    agent=technical_agent
)

manager_task = Task(
    description="""
    Using the information available in the crew's memory,
    prepare a concise customer profile containing:

    - Customer
    - Customer type
    - Number of licensed users
    - Preferred communication method
    - API rate limit

    Do not invent information.
    """,
    expected_output="A concise customer profile.",
    agent=manager_agent
)


# ============================================================
# Attach Memory to the CREW
# ============================================================

crew = Crew(
    agents=[
        support_agent,
        technical_agent,
        manager_agent
    ],

    tasks=[
        support_task,
        technical_task,
        manager_task
    ],

    process=Process.sequential,

    # Shared crew-level memory
    memory=customer_memory,

    verbose=True
)


In [ ]:
result = await crew.kickoff_async()

print("\n========== FINAL RESULT ==========\n")
print(result)

In [ ]:
from crewai import Memory

print(Memory.model_fields["embedder"])
# print(Memory.model_fields["embedder"].annotation)

annotation=Any required=False default=None description='Embedding callable, provider config dict, or None for default OpenAI.'
